#  Synthetic Data Quality Check

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 2)   # 5-fold-cross validation

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [ ]:
final_list = [18, 41, 14, 43, 53, 28, 20, 63, 69, 56, 19, 25, 6, 24, 80, 32, 22, 15, 27, 33, 58,
              46, 29, 64, 62, 17, 47, 13, 44, 9, 49, 55, 3, 35, 67, 54, 12, 7, 39, 36, 4, 79, 59, 52, 5, 57,
              21, 50, 45, 42, 11, 1, 51, 38, 34, 16, 10, 2, 26, 91]
print(len(final_list), final_list)

# 1. Coverage / Support Expansion (Measure k-NN distance)

In [ ]:
# SMOTE
df_res_sm = pd.DataFrame()
for i in final_list:
    df = pd.read_csv('data_newest/ds'+ str(i) +'_new.csv')
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    
    # Make major class as '0' and minor class as '1'
    MAJOR = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() == max(df.iloc[:,-1].value_counts())].index[0] # Moj Label
    minor = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() != max(df.iloc[:,-1].value_counts())].index[0] # min Label    
    df.iloc[:,-1] = df.iloc[:,-1].replace(MAJOR, -100)
    df.iloc[:,-1] = df.iloc[:,-1].replace(minor, 1)
    df.iloc[:,-1] = df.iloc[:,-1].replace(-100, 0)
    df.rename(columns={df.columns[-1]:'NEW_LABEL'}, inplace=True)
#     print('<Modified Class>\n', df.iloc[:,-1].value_counts())
#     print('<Imabalance ratio>\n', "1:{: .3f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
        
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]
    y = df_val.iloc[:, -1]
    
    ##################### For Validation Set #######################
    Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    if i == 33:
        min_strategy = Strategy[1]
    print("<min_strategy>:",min_strategy)
    
    for j in range(len(Strategy)):
        print("==========", "SMOTE_{}".format(Strategy[j]), "==========")
        if min_strategy > Strategy[j]:
            continue         
        else:
            minor_data = df_val[df_val["NEW_LABEL"] == 1]
            need_num = int((len(df_val)-len(minor_data))*Strategy[j]-len(minor_data))
            print("Nedeed Samples:",need_num)

            # Load Data
            df_complete = pd.read_csv(f'data_newest/SMOTE_over/ds{i}_S_{Strategy[j]}_comb.csv')
            df_org = df_complete[:-need_num]
            df_over = df_complete[-need_num:].reset_index(drop=True)

            # Measure Distance
            df_min = df_org[df_org["NEW_LABEL"] == 1].copy()

            X_min = df_min.drop(columns=["NEW_LABEL"]).values
            X_syn = df_over.drop(columns=["NEW_LABEL"], errors="ignore").values

            scaler = StandardScaler()
            X_min_scaled = scaler.fit_transform(X_min)
            X_syn_scaled = scaler.transform(X_syn)

            nn = NearestNeighbors(n_neighbors=5, metric="euclidean")
            nn.fit(X_min_scaled)

            distances, _ = nn.kneighbors(X_syn_scaled)
            distances_avg = []
            for m in range(len(distances)):
                distances_avg.append(np.average(distances[m]))

            df_res_sm[f"ds{i}_SM_{Strategy[j]}"] = [np.mean(distances_avg), np.median(distances_avg), np.std(distances_avg)]
        break   # We use only the first-target-ratio strategy

In [ ]:
df_res_sm.index = ["Mean distance", "Median distance", "Std distance"]
df_res_sm

## 1) With GPT

In [ ]:
# LLM
df_res_lm = pd.DataFrame()
for i in final_list:
    df = pd.read_csv('data_newest/ds'+ str(i) +'_new.csv')
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    
    # Make major class as '0' and minor class as '1'
    MAJOR = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() == max(df.iloc[:,-1].value_counts())].index[0] # Moj Label
    minor = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() != max(df.iloc[:,-1].value_counts())].index[0] # min Label    
    df.iloc[:,-1] = df.iloc[:,-1].replace(MAJOR, -100)
    df.iloc[:,-1] = df.iloc[:,-1].replace(minor, 1)
    df.iloc[:,-1] = df.iloc[:,-1].replace(-100, 0)
    df.rename(columns={df.columns[-1]:'NEW_LABEL'}, inplace=True)
#     print('<Modified Class>\n', df.iloc[:,-1].value_counts())
#     print('<Imabalance ratio>\n', "1:{: .3f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
        
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]
    y = df_val.iloc[:, -1]
    
    ##################### For Validation Set #######################
    Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    if i == 33:
        min_strategy = Strategy[1]
    print("<min_strategy>:",min_strategy)
    
    for j in range(len(Strategy)):
        print("==========", "LLM_{}".format(Strategy[j]), "==========")
        if min_strategy > Strategy[j]:
            continue         
        else:
            minor_data = df_val[df_val["NEW_LABEL"] == 1]
            need_num = int((len(df_val)-len(minor_data))*Strategy[j]-len(minor_data))
            print("Nedeed Samples:",need_num)
            
            # Load Data
            df_complete = pd.read_csv(f'data_newest/LLM_over_re/ds{i}_L_{Strategy[j]}_comb.csv')
            df_complete = df_complete.replace('False', False)  # sometimes False happen
            df_complete = df_complete.replace('FALSE', False)
            df_org = df_complete[:-need_num]
            df_over = df_complete[-need_num:].reset_index(drop=True)

            # Measure Distance
            df_min = df_org[df_org["NEW_LABEL"] == 1].copy()

            X_min = df_min.drop(columns=["NEW_LABEL"]).values
            X_syn = df_over.drop(columns=["NEW_LABEL"], errors="ignore").values

            scaler = StandardScaler()
            X_min_scaled = scaler.fit_transform(X_min)
            X_syn_scaled = scaler.transform(X_syn)

            nn = NearestNeighbors(n_neighbors=5, metric="euclidean")
            nn.fit(X_min_scaled)

            distances, _ = nn.kneighbors(X_syn_scaled)
            distances_avg = []
            for m in range(len(distances)):
                distances_avg.append(np.average(distances[m]))

            df_res_lm[f"ds{i}_LM_{Strategy[j]}"] = [np.mean(distances_avg), np.median(distances_avg), np.std(distances_avg)]
        break   # We use only the first-target-ratio strategy

In [ ]:
df_res_lm.index = ["Mean distance", "Median distance", "Std distance"]
df_res_lm

In [ ]:
# Paired Scatter Plot
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# Convert to numpy arrays
dist_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[1,:] -> median
dist_llm = np.array(df_res_lm.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(dist_smote, dist_llm, alpha=0.7)

# Diagonal line (y = x)
min_val = min(dist_smote.min(), dist_llm.min())
max_val = max(dist_smote.max(), dist_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# Log scale
plt.xscale('log')
plt.yscale('log')

# Labels
plt.xlabel("Mean kNN Distance (SMOTE)")
plt.ylabel("Mean kNN Distance (LLM)")
plt.title("Coverage Comparison (k=5)")

plt.xlim(0.2, 10**(5.5))
plt.ylim(0.2, 10**(5.5))

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# How many datasets where LLM > SMOTE?
num_better = np.sum(dist_llm > dist_smote)
total = len(dist_smote)

print(f"LLM > SMOTE in {num_better}/{total} datasets ({num_better/total:.2%})")

# Mean difference
diff = dist_llm - dist_smote
print("Mean difference:", np.mean(diff))
print("Median difference:", np.median(diff))

## 2) With DEV

In [ ]:
# !!!!!!!!!!!!!!!!!!!!!!!!!!!   DEV   !!!!!!!!!!!!!!!!!!!!!!!!!!
df_res_dev = pd.DataFrame()
for i in final_list:
    df = pd.read_csv('data_newest/ds'+ str(i) +'_new.csv')
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    
    # Make major class as '0' and minor class as '1'
    MAJOR = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() == max(df.iloc[:,-1].value_counts())].index[0] # Moj Label
    minor = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() != max(df.iloc[:,-1].value_counts())].index[0] # min Label    
    df.iloc[:,-1] = df.iloc[:,-1].replace(MAJOR, -100)
    df.iloc[:,-1] = df.iloc[:,-1].replace(minor, 1)
    df.iloc[:,-1] = df.iloc[:,-1].replace(-100, 0)
    df.rename(columns={df.columns[-1]:'NEW_LABEL'}, inplace=True)
#     print('<Modified Class>\n', df.iloc[:,-1].value_counts())
#     print('<Imabalance ratio>\n', "1:{: .3f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
        
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]
    y = df_val.iloc[:, -1]
    
    ##################### For Validation Set #######################
    Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    if i == 33:
        min_strategy = Strategy[1]
    print("<min_strategy>:",min_strategy)
    
    for j in range(len(Strategy)):
        print("==========", "LLM_{}".format(Strategy[j]), "==========")
        if min_strategy > Strategy[j]:
            continue         
        else:
            minor_data = df_val[df_val["NEW_LABEL"] == 1]
            need_num = int((len(df_val)-len(minor_data))*Strategy[j]-len(minor_data))
            print("Nedeed Samples:",need_num)
            
            # Load Data
            df_complete = pd.read_csv(f'data_newest/LLM_2_over_re/ds{i}_L2_{Strategy[j]}_comb.csv')
            df_complete = df_complete.replace('False', False)  # sometimes False happen
            df_complete = df_complete.replace('FALSE', False)
            df_org = df_complete[:-need_num]
            df_over = df_complete[-need_num:].reset_index(drop=True)

            # Measure Distance
            df_min = df_org[df_org["NEW_LABEL"] == 1].copy()

            X_min = df_min.drop(columns=["NEW_LABEL"]).values
            X_syn = df_over.drop(columns=["NEW_LABEL"], errors="ignore").values

            scaler = StandardScaler()
            X_min_scaled = scaler.fit_transform(X_min)
            X_syn_scaled = scaler.transform(X_syn)

            nn = NearestNeighbors(n_neighbors=5, metric="euclidean")
            nn.fit(X_min_scaled)

            distances, _ = nn.kneighbors(X_syn_scaled)
            distances_avg = []
            for m in range(len(distances)):
                distances_avg.append(np.average(distances[m]))

            df_res_dev[f"ds{i}_dev_{Strategy[j]}"] = [np.mean(distances_avg), np.median(distances_avg), np.std(distances_avg)]
        break   # We use only the first-target-ratio strategy

In [ ]:
df_res_dev.index = ["Mean distance", "Median distance", "Std distance"]
df_res_dev

In [ ]:
# Paired Scatter Plot
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# Convert to numpy arrays
dist_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[1,:] -> median
dist_dev = np.array(df_res_dev.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(dist_smote, dist_dev, alpha=0.7)

# Diagonal line (y = x)
min_val = min(dist_smote.min(), dist_dev.min())
max_val = max(dist_smote.max(), dist_dev.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# Log scale
plt.xscale('log')
plt.yscale('log')

# Labels
plt.xlabel("Mean kNN Distance (SMOTE)")
plt.ylabel("Mean kNN Distance (DEV)")
plt.title("Coverage Comparison (k=5)")

plt.xlim(0.2, 10**(5.5))
plt.ylim(0.2, 10**(5.5))

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# How many datasets where LLM > SMOTE?
num_better = np.sum(dist_dev > dist_smote)
total = len(dist_smote)

print(f"LLM > SMOTE in {num_better}/{total} datasets ({num_better/total:.2%})")

# Mean difference
diff = dist_dev - dist_smote
print("Mean difference:", np.mean(diff))
print("Median difference:", np.median(diff))

## 3) Together

In [ ]:
# Paired Scatter Plot
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# Convert to numpy arrays
dist_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[1,:] -> median
dist_llm = np.array(df_res_lm.iloc[0,:])
dist_dev = np.array(df_res_dev.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(dist_smote, dist_llm, alpha=0.7, marker='^', label='GPT')
plt.scatter(dist_smote, dist_dev, alpha=0.7, label='DEV')

# Diagonal line (y = x)
min_val = min(dist_smote.min(), dist_llm.min())
max_val = max(dist_smote.max(), dist_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# Log scale
plt.xscale('log')
plt.yscale('log')

# Labels
plt.xlabel("Mean kNN Distance (SMOTE)")
plt.ylabel("Mean kNN Distance (LLM)")
plt.title("Coverage Comparison (k=5)")

plt.xlim(0.2, 10**(5.5))
plt.ylim(0.2, 10**(5.5))

plt.grid(True)
plt.tight_layout()
plt.legend()
plt.show()

# 2. Classifier Uncertainty

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn import svm
from sklearn import neighbors
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:
def compute_uncertainty(df_syn):
    X_syn = df_syn.drop(columns=["NEW_LABEL"], errors="ignore").values
    X_syn_scaled = scaler.transform(X_syn)

    # Predict probability for minority class
    prob = clf.predict_proba(X_syn_scaled)[:, 1]

    # Uncertainty = closeness to 0.5
    uncertainty = np.abs(prob - 0.5)

    return uncertainty

In [ ]:
# SMOTE
df_res_sm = pd.DataFrame()
for i in final_list:
    df = pd.read_csv('data_newest/ds'+ str(i) +'_new.csv')
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    
    # Make major class as '0' and minor class as '1'
    MAJOR = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() == max(df.iloc[:,-1].value_counts())].index[0] # Moj Label
    minor = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() != max(df.iloc[:,-1].value_counts())].index[0] # min Label    
    df.iloc[:,-1] = df.iloc[:,-1].replace(MAJOR, -100)
    df.iloc[:,-1] = df.iloc[:,-1].replace(minor, 1)
    df.iloc[:,-1] = df.iloc[:,-1].replace(-100, 0)
    df.rename(columns={df.columns[-1]:'NEW_LABEL'}, inplace=True)
#     print('<Modified Class>\n', df.iloc[:,-1].value_counts())
#     print('<Imabalance ratio>\n', "1:{: .3f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
        
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]
    y = df_val.iloc[:, -1]
    
    ##################### For Validation Set #######################
    Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    if i == 33:
        min_strategy = Strategy[1]
    print("<min_strategy>:",min_strategy)
    
    for j in range(len(Strategy)):
        print("==========", "SMOTE_{}".format(Strategy[j]), "==========")
        if min_strategy > Strategy[j]:
            continue         
        else:     
            minor_data = df_val[df_val["NEW_LABEL"] == 1]
            need_num = int((len(df_val)-len(minor_data))*Strategy[j]-len(minor_data))
            print("Nedeed Samples:",need_num)

            # Load Data
            df_complete = pd.read_csv(f'data_newest/SMOTE_over/ds{i}_S_{Strategy[j]}_comb.csv')
            df_org = df_complete[:-need_num]
            df_over = df_complete[-need_num:].reset_index(drop=True)

            # Measure Uncertainty
            X = df_org.drop(columns=["NEW_LABEL"]).values
            y = df_org["NEW_LABEL"].values

            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)

            clf = DecisionTreeClassifier(min_samples_split=10, min_samples_leaf=5, random_state=42) 
            clf = svm.SVC(probability=True, random_state=42)
            clf = neighbors.KNeighborsClassifier(n_neighbors=3, weights='distance')  # weights='distance'
            clf = LGBMClassifier(random_state=42)
            clf.fit(X_scaled, y)

            unc = compute_uncertainty(df_over)

            df_res_sm[f"ds{i}_SM_{Strategy[j]}"] = [np.mean(unc), np.median(unc), np.std(unc)]
        break   # We use only the first-target-ratio strategy

In [ ]:
df_res_sm.index = ["Mean unc", "Median unc", "Std unc"]
df_res_sm

## 1) With GPT

In [ ]:
# LLM
df_res_lm = pd.DataFrame()
for i in final_list:
    df = pd.read_csv('data_newest/ds'+ str(i) +'_new.csv')
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    
    # Make major class as '0' and minor class as '1'
    MAJOR = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() == max(df.iloc[:,-1].value_counts())].index[0] # Moj Label
    minor = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() != max(df.iloc[:,-1].value_counts())].index[0] # min Label    
    df.iloc[:,-1] = df.iloc[:,-1].replace(MAJOR, -100)
    df.iloc[:,-1] = df.iloc[:,-1].replace(minor, 1)
    df.iloc[:,-1] = df.iloc[:,-1].replace(-100, 0)
    df.rename(columns={df.columns[-1]:'NEW_LABEL'}, inplace=True)
#     print('<Modified Class>\n', df.iloc[:,-1].value_counts())
#     print('<Imabalance ratio>\n', "1:{: .3f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
        
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]
    y = df_val.iloc[:, -1]
    
    ##################### For Validation Set #######################
    Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    if i == 33:
        min_strategy = Strategy[1]
    print("<min_strategy>:",min_strategy)
    
    for j in range(len(Strategy)):
        print("==========", "LLM_{}".format(Strategy[j]), "==========")
        if min_strategy > Strategy[j]:
            continue         
        else:     
            minor_data = df_val[df_val["NEW_LABEL"] == 1]
            need_num = int((len(df_val)-len(minor_data))*Strategy[j]-len(minor_data))
            print("Nedeed Samples:",need_num)

            # Load Data
            df_complete = pd.read_csv(f'data_newest/LLM_over_re/ds{i}_L_{Strategy[j]}_comb.csv')
            df_complete = df_complete.replace('False', False)  # sometimes False happen
            df_complete = df_complete.replace('FALSE', False)
            df_org = df_complete[:-need_num]
            df_over = df_complete[-need_num:].reset_index(drop=True)

            # Measure Uncertainty
            X = df_org.drop(columns=["NEW_LABEL"]).values
            y = df_org["NEW_LABEL"].values

            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)

            clf = DecisionTreeClassifier(min_samples_split=10, min_samples_leaf=5, random_state=42) 
            clf = svm.SVC(probability=True, random_state=42)
            clf = neighbors.KNeighborsClassifier(n_neighbors=3, weights='distance')  # weights='distance'
            clf = LGBMClassifier(random_state=42)
            clf.fit(X_scaled, y)

            unc = compute_uncertainty(df_over)

            df_res_lm[f"ds{i}_LM_{Strategy[j]}"] = [np.mean(unc), np.median(unc), np.std(unc)]
        break   # We use only the first-target-ratio strategy

In [ ]:
df_res_lm.index = ["Mean unc", "Median unc", "Std unc"]
df_res_lm

### Decision Tree

In [ ]:
# Paired Scatter Plot

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

import numpy as np
import matplotlib.pyplot as plt

# Convert to numpy arrays
unc_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[,:] -> median
unc_llm = np.array(df_res_lm.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(unc_smote, unc_llm, alpha=0.7)

# Diagonal line (y = x)
min_val = min(unc_smote.min(), unc_llm.min())
max_val = max(unc_smote.max(), unc_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# # Log scale
# plt.xscale('log')
# plt.yscale('log')

# Labels
plt.xlabel("Mean Uncertainty (SMOTE)")
plt.ylabel("Mean Uncertainty (LLM)")
plt.title("Uncertainty Comparison (DT)")

# plt.xlim(min_val, max_val)
# plt.ylim(min_val, max_val)

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# How many datasets where LLM < SMOTE? (Smaller is better)
num_better = np.sum(unc_llm < unc_smote)
num_even = np.sum(unc_llm == unc_smote)
num_worse = np.sum(unc_llm > unc_smote)
total = len(unc_smote)

print(f"LLM < SMOTE in {num_better}/{total} datasets ({num_better/total:.2%})")
print(f"LLM = SMOTE in {num_even}/{total} datasets ({num_even/total:.2%})")
print(f"LLM > SMOTE in {num_worse}/{total} datasets ({num_worse/total:.2%})")

# Mean difference
diff = unc_smote - unc_llm
print("Mean difference:", np.mean(diff))
print("Median difference:", np.median(diff))

In [ ]:
datasets = range(1,61)

plt.plot(datasets, diff, marker='s', label='difference')

# Formatting
# plt.yscale('log')
plt.xlabel('Imbalance Ratio (IR)')
plt.ylabel('Score Margin')
plt.title('Performance with Varying IRs')
plt.xticks(datasets)
# plt.ylim(-0.02, 0.08)
plt.grid(True, which='both', axis='y', linestyle='--', alpha=0.6)
plt.legend(fontsize=13)
plt.tight_layout()
plt.savefig('performance_with_values.png')
plt.show()

### SVM

In [ ]:
# Paired Scatter Plot

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

import numpy as np
import matplotlib.pyplot as plt

# Convert to numpy arrays
unc_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[,:] -> median
unc_llm = np.array(df_res_lm.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(unc_smote, unc_llm, alpha=0.7)

# Diagonal line (y = x)
min_val = min(unc_smote.min(), unc_llm.min())
max_val = max(unc_smote.max(), unc_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# # Log scale
# plt.xscale('log')
# plt.yscale('log')

# Labels
plt.xlabel("Mean Uncertainty (SMOTE)")
plt.ylabel("Mean Uncertainty (LLM)")
plt.title("Uncertainty Comparison (SVM)")

# plt.xlim(min_val, max_val)
# plt.ylim(min_val, max_val)

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# How many datasets where LLM < SMOTE? (Smaller is better)
num_better = np.sum(unc_llm < unc_smote)
num_even = np.sum(unc_llm == unc_smote)
num_worse = np.sum(unc_llm > unc_smote)
total = len(unc_smote)

print(f"LLM < SMOTE in {num_better}/{total} datasets ({num_better/total:.2%})")
print(f"LLM = SMOTE in {num_even}/{total} datasets ({num_even/total:.2%})")
print(f"LLM > SMOTE in {num_worse}/{total} datasets ({num_worse/total:.2%})")

# Mean difference
diff = unc_smote - unc_llm
print("Mean difference:", np.mean(diff))
print("Median difference:", np.median(diff))

In [ ]:
datasets = range(1,61)

plt.plot(datasets, diff, marker='s', label='difference')

# Formatting
# plt.yscale('log')
plt.xlabel('Imbalance Ratio (IR)')
plt.ylabel('Score Margin')
plt.title('Performance with Varying IRs')
plt.xticks(datasets)
# plt.ylim(-0.02, 0.08)
plt.grid(True, which='both', axis='y', linestyle='--', alpha=0.6)
plt.legend(fontsize=13)
plt.tight_layout()
plt.savefig('performance_with_values.png')
plt.show()

### kNN

In [ ]:
# Paired Scatter Plot

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

import numpy as np
import matplotlib.pyplot as plt

# Convert to numpy arrays
unc_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[,:] -> median
unc_llm = np.array(df_res_lm.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(unc_smote, unc_llm, alpha=0.7)

# Diagonal line (y = x)
min_val = min(unc_smote.min(), unc_llm.min())
max_val = max(unc_smote.max(), unc_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# # Log scale
# plt.xscale('log')
# plt.yscale('log')

# Labels
plt.xlabel("Mean Uncertainty (SMOTE)")
plt.ylabel("Mean Uncertainty (LLM)")
plt.title("Uncertainty Comparison (kNN)")

# plt.xlim(min_val, max_val)
# plt.ylim(min_val, max_val)

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# How many datasets where LLM < SMOTE? (Smaller is better)
num_better = np.sum(unc_llm < unc_smote)
num_even = np.sum(unc_llm == unc_smote)
num_worse = np.sum(unc_llm > unc_smote)
total = len(unc_smote)

print(f"LLM < SMOTE in {num_better}/{total} datasets ({num_better/total:.2%})")
print(f"LLM = SMOTE in {num_even}/{total} datasets ({num_even/total:.2%})")
print(f"LLM > SMOTE in {num_worse}/{total} datasets ({num_worse/total:.2%})")

# Mean difference
diff = unc_smote - unc_llm
print("Mean difference:", np.mean(diff))
print("Median difference:", np.median(diff))

In [ ]:
datasets = range(1,61)

plt.plot(datasets, diff, marker='s', label='difference')

# Formatting
# plt.yscale('log')
plt.xlabel('Imbalance Ratio (IR)')
plt.ylabel('Score Margin')
plt.title('Performance with Varying IRs')
plt.xticks(datasets)
# plt.ylim(-0.02, 0.08)
plt.grid(True, which='both', axis='y', linestyle='--', alpha=0.6)
plt.legend(fontsize=13)
plt.tight_layout()
plt.savefig('performance_with_values.png')
plt.show()

### LightGBM

In [ ]:
# Paired Scatter Plot

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

import numpy as np
import matplotlib.pyplot as plt

# Convert to numpy arrays
unc_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[,:] -> median
unc_llm = np.array(df_res_lm.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(unc_smote, unc_llm, alpha=0.7)

# Diagonal line (y = x)
min_val = min(unc_smote.min(), unc_llm.min())
max_val = max(unc_smote.max(), unc_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# # Log scale
# plt.xscale('log')
# plt.yscale('log')

# Labels
plt.xlabel("Mean Uncertainty (SMOTE)")
plt.ylabel("Mean Uncertainty (LLM)")
plt.title("Uncertainty Comparison (LG)")

# plt.xlim(min_val, max_val)
# plt.ylim(min_val, max_val)

plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# How many datasets where LLM < SMOTE? (Smaller is better)
num_better = np.sum(unc_llm < unc_smote)
num_even = np.sum(unc_llm == unc_smote)
num_worse = np.sum(unc_llm > unc_smote)
total = len(unc_smote)

print(f"LLM < SMOTE in {num_better}/{total} datasets ({num_better/total:.2%})")
print(f"LLM = SMOTE in {num_even}/{total} datasets ({num_even/total:.2%})")
print(f"LLM > SMOTE in {num_worse}/{total} datasets ({num_worse/total:.2%})")

# Mean difference
diff = unc_smote - unc_llm
print("Mean difference:", np.mean(diff))
print("Median difference:", np.median(diff))

In [ ]:
datasets = range(1,61)

plt.plot(datasets, diff, marker='s', label='difference')

# Formatting
# plt.yscale('log')
plt.xlabel('Imbalance Ratio (IR)')
plt.ylabel('Score Margin')
plt.title('Performance with Varying IRs')
plt.xticks(datasets)
# plt.ylim(-0.02, 0.08)
plt.grid(True, which='both', axis='y', linestyle='--', alpha=0.6)
plt.legend(fontsize=13)
plt.tight_layout()
plt.savefig('performance_with_values.png')
plt.show()

## 2) With DEV

In [ ]:
# !!!!!!!!!!!!!!!!!!!!!! DEV !!!!!!!!!!!!!!!!!!!!!!!!!!!
df_res_dev = pd.DataFrame()
for i in final_list:
    df = pd.read_csv('data_newest/ds'+ str(i) +'_new.csv')
    print('+'*35, '{}th Dataset'.format(i), '+'*35)
    
    # Make major class as '0' and minor class as '1'
    MAJOR = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() == max(df.iloc[:,-1].value_counts())].index[0] # Moj Label
    minor = df.iloc[:,-1].value_counts()[df.iloc[:,-1].value_counts() != max(df.iloc[:,-1].value_counts())].index[0] # min Label    
    df.iloc[:,-1] = df.iloc[:,-1].replace(MAJOR, -100)
    df.iloc[:,-1] = df.iloc[:,-1].replace(minor, 1)
    df.iloc[:,-1] = df.iloc[:,-1].replace(-100, 0)
    df.rename(columns={df.columns[-1]:'NEW_LABEL'}, inplace=True)
#     print('<Modified Class>\n', df.iloc[:,-1].value_counts())
#     print('<Imabalance ratio>\n', "1:{: .3f}".format(df.iloc[:,-1].value_counts()[1]/df.iloc[:,-1].value_counts()[0]))
        
    ##################### Validation:Test = 70:30 #######################
    df_val, df_test = train_test_split(df, test_size=0.3, random_state=100, stratify=df.iloc[:,-1])
    X = df_val.iloc[:, :-1]
    y = df_val.iloc[:, -1]
    
    ##################### For Validation Set #######################
    Strategy = [0.2, 0.4, 0.6, 0.8, 1.0]
    ind = int((y.value_counts()[1]/y.value_counts()[0])//0.2)
    min_strategy = Strategy[ind]
    if i == 33:
        min_strategy = Strategy[1]
    print("<min_strategy>:",min_strategy)
    
    for j in range(len(Strategy)):
        print("==========", "LLM_{}".format(Strategy[j]), "==========")
        if min_strategy > Strategy[j]:
            continue         
        else:     
            minor_data = df_val[df_val["NEW_LABEL"] == 1]
            need_num = int((len(df_val)-len(minor_data))*Strategy[j]-len(minor_data))
            print("Nedeed Samples:",need_num)

            # Load Data
            df_complete = pd.read_csv(f'data_newest/LLM_2_over_re/ds{i}_L2_{Strategy[j]}_comb.csv')
            df_complete = df_complete.replace('False', False)  # sometimes False happen
            df_complete = df_complete.replace('FALSE', False)
            df_org = df_complete[:-need_num]
            df_over = df_complete[-need_num:].reset_index(drop=True)

            # Measure Uncertainty
            X = df_org.drop(columns=["NEW_LABEL"]).values
            y = df_org["NEW_LABEL"].values

            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X)

            clf = DecisionTreeClassifier(min_samples_split=10, min_samples_leaf=5, random_state=42) 
            clf = svm.SVC(probability=True, random_state=42)
            clf = neighbors.KNeighborsClassifier(n_neighbors=3, weights='distance')  # weights='distance'
            clf = LGBMClassifier(random_state=42)
            clf.fit(X_scaled, y)

            unc = compute_uncertainty(df_over)

            df_res_dev[f"ds{i}_DEV_{Strategy[j]}"] = [np.mean(unc), np.median(unc), np.std(unc)]
        break   # We use only the first-target-ratio strategy

In [ ]:
df_res_dev.index = ["Mean unc", "Median unc", "Std unc"]
df_res_dev

## DT

In [ ]:
# Paired Scatter Plot

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# Convert to numpy arrays
unc_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[,:] -> median
unc_llm = np.array(df_res_lm.iloc[0,:])
unc_dev = np.array(df_res_dev.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(unc_smote, unc_llm, alpha=0.7, color = 'tab:green', marker='^', label = 'GPT')
plt.scatter(unc_smote, unc_dev, alpha=0.7, color = 'tab:red', label = 'DEV')

# Diagonal line (y = x)
min_val = min(unc_smote.min(), unc_llm.min())
max_val = max(unc_smote.max(), unc_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# # Log scale
# plt.xscale('log')
# plt.yscale('log')

# Labels
plt.xlabel("Mean Uncertainty (SMOTE)")
plt.ylabel("Mean Uncertainty (LLM)")
plt.title("Uncertainty Comparison (DT)")

# plt.xlim(min_val, max_val)
# plt.ylim(min_val, max_val)

plt.grid(True)
plt.tight_layout()
plt.legend()
plt.show()

## SVM

In [ ]:
# Paired Scatter Plot

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# Convert to numpy arrays
unc_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[,:] -> median
unc_llm = np.array(df_res_lm.iloc[0,:])
unc_dev = np.array(df_res_dev.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(unc_smote, unc_llm, alpha=0.7, color = 'tab:green', marker='^', label = 'GPT')
plt.scatter(unc_smote, unc_dev, alpha=0.7, color = 'tab:red', label = 'DEV')

# Diagonal line (y = x)
min_val = min(unc_smote.min(), unc_llm.min())
max_val = max(unc_smote.max(), unc_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# # Log scale
# plt.xscale('log')
# plt.yscale('log')

# Labels
plt.xlabel("Mean Uncertainty (SMOTE)")
plt.ylabel("Mean Uncertainty (LLM)")
plt.title("Uncertainty Comparison (SVM)")

# plt.xlim(min_val, max_val)
# plt.ylim(min_val, max_val)

plt.grid(True)
plt.tight_layout()
plt.legend()
plt.show()

## kNN

In [ ]:
# Paired Scatter Plot

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# Convert to numpy arrays
unc_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[,:] -> median
unc_llm = np.array(df_res_lm.iloc[0,:])
unc_dev = np.array(df_res_dev.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(unc_smote, unc_llm, alpha=0.7, color = 'tab:green', marker='^', label = 'GPT')
plt.scatter(unc_smote, unc_dev, alpha=0.7, color = 'tab:red', label = 'DEV')

# Diagonal line (y = x)
min_val = min(unc_smote.min(), unc_llm.min())
max_val = max(unc_smote.max(), unc_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# # Log scale
# plt.xscale('log')
# plt.yscale('log')

# Labels
plt.xlabel("Mean Uncertainty (SMOTE)")
plt.ylabel("Mean Uncertainty (LLM)")
plt.title("Uncertainty Comparison (kNN)")

# plt.xlim(min_val, max_val)
# plt.ylim(min_val, max_val)

plt.grid(True)
plt.tight_layout()
plt.legend()
plt.show()

## LGBM

In [ ]:
# Paired Scatter Plot

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 200
plt.rcParams['axes.titlesize'] = 20  
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['axes.labelsize'] = 20  
plt.rcParams['font.size'] = 15

# Convert to numpy arrays
unc_smote = np.array(df_res_sm.iloc[0,:])       # iloc[0,:] -> mean  iloc[,:] -> median
unc_llm = np.array(df_res_lm.iloc[0,:])
unc_dev = np.array(df_res_dev.iloc[0,:])

# -----------------------------
# Scatter plot
# -----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(unc_smote, unc_llm, alpha=0.7, color = 'tab:green', marker='^', label = 'GPT')
plt.scatter(unc_smote, unc_dev, alpha=0.7, color = 'tab:red', label = 'DEV')

# Diagonal line (y = x)
min_val = min(unc_smote.min(), unc_llm.min())
max_val = max(unc_smote.max(), unc_llm.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle='--')

# # Log scale
# plt.xscale('log')
# plt.yscale('log')

# Labels
plt.xlabel("Mean Uncertainty (SMOTE)")
plt.ylabel("Mean Uncertainty (LLM)")
plt.title("Uncertainty Comparison (LG)")

# plt.xlim(min_val, max_val)
# plt.ylim(min_val, max_val)

plt.grid(True)
plt.tight_layout()
plt.legend()
plt.show()